# EEG student review and segment preparation

This notebook provides a graphical workflow for reviewing BrainVision EEG recordings (`.vhdr`, `.eeg`, `.vmrk`). The original files are never modified.

Run Steps 1–5 in order:
1. Select one or more `.vhdr` files and an output folder in Windows dialogs.
2. Load the helper functions.
3. Choose one queued recording and create an untouched master plus a 250 Hz, 0.5–50 Hz filtered copy.
4. Inspect all channels and annotations using the stacked Plotly average-reference display.
5. Use the GUI to standardize/insert annotations, rename channels, mark bad channels, confirm the selected range in Plotly, and save.

**Important:** Step 4 shows the 250 Hz, 0.5–50 Hz filtered, average-referenced signal in physical units so you can select processing sections. Global z-scoring is applied only to the sections selected for final export. Unannotated time is unreviewed, not automatically clean.


In [ ]:
%pip install h5py

In [ ]:
%pip install mne

In [2]:
# Setup: install missing packages if needed.
# The technical lead should run this once before the student begins.
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "plotly": "plotly",
    "scipy": "scipy",
    "mne": "mne",
    "PyQt6": "PyQt6",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    print("Installation complete. Restart the notebook kernel, then run this cell again.")
else:
    get_ipython().run_line_magic("gui", "qt5")
    print("GUI packages are ready.")


ERROR:root:
    Could not load requested Qt binding. Please ensure that
    PyQt4 >= 4.7, PyQt5, PyQt6, PySide >= 1.0.3, PySide2, or
    PySide6 is available, and only one is imported per session.

    Currently-imported Qt library:                              None
    PyQt5 available (requires QtCore, QtGui, QtSvg, QtWidgets): False
    PyQt6 available (requires QtCore, QtGui, QtSvg, QtWidgets): True
    PySide2 installed:                                          False
    PySide6 installed:                                          False
    Tried to load:                                              ['pyqt5']
    


GUI packages are ready.


In [4]:
# Step 1: select patient recordings and the review output folder.
# Running this cell opens normal Windows selection dialogs.

from pathlib import Path
from PyQt6.QtWidgets import QApplication, QFileDialog

_app = QApplication.instance() or QApplication([])
VHDR_FILES, _ = QFileDialog.getOpenFileNames(
    None,
    "Select one or more BrainVision EEG header files",
    "",
    "BrainVision header files (*.vhdr)",
)
if not VHDR_FILES:
    raise RuntimeError("No .vhdr files were selected.")

EXPORT_FOLDER = QFileDialog.getExistingDirectory(
    None,
    "Choose the folder where reviewed data will be saved",
)
if not EXPORT_FOLDER:
    raise RuntimeError("No output folder was selected.")

print(f"Selected {len(VHDR_FILES)} recording(s):")
for number, path in enumerate(VHDR_FILES, start=1):
    print(f"  {number}. {Path(path).name}")
print("Review output folder:", EXPORT_FOLDER)


Selected 2 recording(s):
  1. 20260805OR.vhdr
  2. 20260805preor.vhdr
Review output folder: C:/Users/zhouz/OneDrive - UC Irvine/Documents/Anes_Liveamp8_Data/Reorganized_liveamp8_data


## 2. Functions

In [8]:
# Step 2: functions to load, plot, zoom, and export.

from pathlib import Path
import csv
import json
import math
import numpy as np
import plotly.graph_objects as go
from scipy.io import savemat


def _parse_key_value_file(path):
    sections = {}
    current = None
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith(";"):
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip().lower()
                sections[current] = {}
                continue
            if current is None or "=" not in line:
                continue
            key, value = line.split("=", 1)
            sections[current][key.strip()] = value.strip()
    return sections


def load_brainvision(vhdr_file):
    vhdr_path = Path(vhdr_file)
    folder = vhdr_path.parent
    header = _parse_key_value_file(vhdr_path)
    common = header.get("common infos", {})
    binary = header.get("binary infos", {})
    channels = header.get("channel infos", {})

    data_file = common["DataFile"]
    data_format = common.get("DataFormat", "").upper()
    orientation = common.get("DataOrientation", "").upper()
    n_channels = int(common["NumberOfChannels"])
    srate = 1_000_000 / float(common["SamplingInterval"])
    binary_format = binary.get("BinaryFormat", "").upper()

    if data_format != "BINARY":
        raise ValueError(f"This beginner notebook expects DataFormat=BINARY, found {data_format!r}.")
    if orientation != "MULTIPLEXED":
        raise ValueError(f"This notebook expects DataOrientation=MULTIPLEXED, found {orientation!r}.")

    dtype_map = {
        "IEEE_FLOAT_32": np.float32,
        "INT_16": np.int16,
        "UINT_16": np.uint16,
    }
    if binary_format not in dtype_map:
        raise ValueError(f"Unsupported BinaryFormat: {binary_format!r}")
    dtype = dtype_map[binary_format]

    labels = []
    scales = []
    for ch in range(1, n_channels + 1):
        entry = channels.get(f"Ch{ch}", f"Ch {ch},,1,")
        parts = entry.split(",")
        labels.append(parts[0].strip() or f"Ch {ch}")
        scale = float(parts[2]) if len(parts) >= 3 and parts[2].strip() else 1.0
        scales.append(scale)

    eeg_path = folder / data_file
    raw = np.fromfile(eeg_path, dtype=dtype)
    if raw.size % n_channels != 0:
        raise ValueError("The .eeg file length is not divisible by the number of channels.")
    n_samples = raw.size // n_channels
    data = raw.reshape(n_samples, n_channels).T.astype(float)
    data *= np.asarray(scales)[:, None]

    marker_path = folder / f"{vhdr_path.stem}.vmrk"
    markers = []
    if marker_path.exists():
        marker_info = _parse_key_value_file(marker_path).get("marker infos", {})
        for key, value in marker_info.items():
            parts = value.split(",")
            if len(parts) >= 3:
                marker_type = parts[0].strip()
                label = parts[1].strip().replace(r"\1", ",")
                sample = int(float(parts[2]))
                if marker_type.lower() == "comment" and label:
                    markers.append({"label": label, "sample": sample, "time_sec": (sample - 1) / srate})

    return {
        "vhdr_file": str(vhdr_path),
        "eeg_file": str(eeg_path),
        "marker_file": str(marker_path),
        "data": data,
        "labels": labels,
        "srate": srate,
        "n_samples": n_samples,
        "duration_sec": n_samples / srate,
        "markers": markers,
    }


def _downsample_indices(n_samples, max_points):
    step = max(1, math.ceil(n_samples / max_points))
    return np.arange(0, n_samples, step)


def _reference_for_view(y, plot_mode):
    if plot_mode == "raw":
        return y, "Amplitude"

    if plot_mode == "average_reference":
        return y - np.nanmean(y, axis=0, keepdims=True), "Average-referenced amplitude"
    else:
        raise ValueError("plot_mode must be 'raw' or 'average_reference'.")


def plot_raw(eeg, start_sec=None, end_sec=None, channels="all", max_points=6000, plot_mode="average_reference"):
    data = eeg["data"]
    labels = eeg["labels"]
    srate = eeg["srate"]
    if start_sec is None:
        start_sec = 0
    if end_sec is None:
        end_sec = eeg["duration_sec"]
    start_sample = max(0, int(round(start_sec * srate)))
    end_sample = min(eeg["n_samples"], int(round(end_sec * srate)))

    if channels == "all":
        channel_indices = list(range(data.shape[0]))
    else:
        channel_indices = [labels.index(ch) if isinstance(ch, str) else int(ch) for ch in channels]

    sample_indices = start_sample + _downsample_indices(end_sample - start_sample, max_points)
    t = sample_indices / srate / 60.0
    y_raw = data[np.ix_(channel_indices, sample_indices)]
    y, ylabel = _reference_for_view(y_raw, plot_mode)

    robust = np.nanpercentile(np.abs(y - np.nanmedian(y)), 95)
    spacing = robust * 4 if np.isfinite(robust) and robust > 0 else 1.0
    offsets = np.arange(len(channel_indices))[::-1] * spacing

    fig = go.Figure()
    for row, ch in enumerate(channel_indices):
        fig.add_trace(go.Scattergl(
            x=t,
            y=y[row] + offsets[row],
            mode="lines",
            name=labels[ch],
            line={"width": 1},
        ))

    top = float(np.nanmax(y + offsets[:, None]))
    bottom = float(np.nanmin(y + offsets[:, None]))
    height = top - bottom if top > bottom else 1.0
    label_y = top + 0.05 * height

    for marker in eeg["markers"]:
        x = marker["time_sec"] / 60.0
        if start_sec <= marker["time_sec"] <= end_sec:
            fig.add_vline(x=x, line_color="red", line_dash="dash", opacity=0.65)
            fig.add_annotation(
                x=x,
                y=label_y,
                text=marker["label"],
                showarrow=False,
                textangle=-45,
                font={"color": "red", "size": 11},
                yanchor="bottom",
            )

    fig.update_layout(
        title=f"EEG with marker labels ({plot_mode})",
        xaxis_title="Time (minutes)",
        yaxis={"title": ylabel, "tickmode": "array", "tickvals": offsets, "ticktext": [labels[ch] for ch in channel_indices]},
        hovermode="x unified",
        height=max(520, 90 * len(channel_indices)),
        margin={"l": 80, "r": 30, "t": 100, "b": 50},
    )
    fig.update_yaxes(range=[bottom - 0.05 * height, top + 0.25 * height])
    fig.show()


def export_segment(eeg, start_sec, end_sec, segment_name, export_folder=EXPORT_FOLDER):
    export_path = Path(export_folder)
    export_path.mkdir(parents=True, exist_ok=True)

    srate = eeg["srate"]
    start_sample = max(0, int(round(start_sec * srate)))
    end_sample = min(eeg["n_samples"], int(round(end_sec * srate)))
    if end_sample <= start_sample:
        raise ValueError("End time must be after start time.")

    data_segment = eeg["data"][:, start_sample:end_sample]
    time_sec = np.arange(start_sample, end_sample) / srate
    segment_markers = [m for m in eeg["markers"] if start_sec <= m["time_sec"] <= end_sec]

    safe_name = "".join(c if c.isalnum() or c in "-_" else "_" for c in segment_name).strip("_")
    base = export_path / safe_name

    with open(base.with_suffix(".csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["time_sec", *eeg["labels"]])
        for i in range(data_segment.shape[1]):
            writer.writerow([time_sec[i], *data_segment[:, i]])

    np.savez_compressed(
        base.with_suffix(".npz"),
        data=data_segment,
        time_sec=time_sec,
        srate=srate,
        channel_labels=np.array(eeg["labels"], dtype=object),
        markers=np.array(segment_markers, dtype=object),
    )

    savemat(base.with_suffix(".mat"), {
        "data": data_segment,
        "time_sec": time_sec,
        "srate": srate,
        "channel_labels": np.array(eeg["labels"], dtype=object),
        "marker_labels": np.array([m["label"] for m in segment_markers], dtype=object),
        "marker_times_sec": np.array([m["time_sec"] for m in segment_markers]),
    })

    metadata = {
        "source_vhdr_file": eeg["vhdr_file"],
        "source_eeg_file": eeg["eeg_file"],
        "start_sec": start_sec,
        "end_sec": end_sec,
        "start_sample": start_sample,
        "end_sample_exclusive": end_sample,
        "srate": srate,
        "channel_labels": eeg["labels"],
        "markers_in_segment": segment_markers,
        "files": [str(base.with_suffix(ext)) for ext in [".csv", ".npz", ".mat", ".json"]],
    }
    with open(base.with_suffix(".json"), "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print("Saved segment files:")
    print(base.with_suffix(".csv"))
    print(base.with_suffix(".npz"))
    print(base.with_suffix(".mat"))
    print(base.with_suffix(".json"))

# ---------- Beginner GUI review helpers ----------

import hashlib
import re
from datetime import datetime, timezone

import mne
from PyQt6.QtCore import Qt
from PyQt6.QtWidgets import (
    QApplication,
    QAbstractItemView,
    QCheckBox,
    QComboBox,
    QDialog,
    QDialogButtonBox,
    QDoubleSpinBox,
    QFileDialog,
    QFormLayout,
    QGroupBox,
    QHBoxLayout,
    QInputDialog,
    QLabel,
    QLineEdit,
    QListWidget,
    QMessageBox,
    QPushButton,
    QTableWidget,
    QTableWidgetItem,
    QVBoxLayout,
)


STANDARD_LABELS = [
    "phase_preop",
    "phase_induction",
    "phase_maintenance",
    "phase_emergence",
    "phase_postop",
    "propofol_drip",
    "sevoflurane",
    "lidocaine_bolus",
    "propofol_bolus",
    "ketamine_bolus",
    "midazolam_bolus",
    "LOC",
    "Intubation",
    "Incision",
    "fentanyl_bolus",
    "morphine_bolus",
    "hydromorphone_bolus",
    "dexmedetomidine_bolus",
    "dexmedetomidine_bolus_10",
    "dexmedetomidine_bolus_20",
    "dexmedetomidine_bolus_40",
    "dexmedetomidine_drip",
    "lidocaine_bolus",
    "rocuronium_bolus",
    "Loss_of_responsiveness",
    "Return_of_responsiveness",
    "extubation",
    "ROC",
    "close",
    "BAD_eye_blink",
    "BAD_ECG",
    "BAD_movement",
    "BAD_electrocautery",
    "BAD_electrode_pop",
    "BAD_flat_channel",
    "BAD_line_noise",
    "BAD_other",
    "UNMAPPED_review_needed",
]

TARGET_SAMPLING_RATE_HZ = 250.0


def _qt_app():
    """Return the existing Qt application, or create one."""
    return QApplication.instance() or QApplication([])


def choose_brainvision_files_and_output():
    """Open native Windows dialogs for source files and output folder."""
    _qt_app()
    files, _ = QFileDialog.getOpenFileNames(
        None,
        "Select one or more BrainVision EEG header files",
        "",
        "BrainVision header files (*.vhdr)",
    )
    if not files:
        raise RuntimeError("No .vhdr files were selected.")

    output = QFileDialog.getExistingDirectory(
        None,
        "Choose the folder where reviewed data will be saved",
    )
    if not output:
        raise RuntimeError("No output folder was selected.")
    return [str(Path(path)) for path in files], str(Path(output))


def _copy_annotations(annotations):
    return mne.Annotations(
        onset=annotations.onset.copy(),
        duration=annotations.duration.copy(),
        description=annotations.description.copy(),
        orig_time=annotations.orig_time,
        ch_names=list(annotations.ch_names),
    )


def prepare_gui_recording(vhdr_files):
    """Choose one queued file and create master, filtered, and display copies."""
    _qt_app()
    names = [Path(path).name for path in vhdr_files]
    chosen, accepted = QInputDialog.getItem(
        None,
        "Choose recording",
        "Recording to review:",
        names,
        0,
        False,
    )
    if not accepted:
        raise RuntimeError("Recording selection was cancelled.")

    source = Path(vhdr_files[names.index(chosen)])
    master = mne.io.read_raw_brainvision(source, preload=True, verbose="ERROR")
    if master.info["sfreq"] < TARGET_SAMPLING_RATE_HZ:
        raise ValueError(
            f"Source sampling frequency must be at least {TARGET_SAMPLING_RATE_HZ:g} Hz for downsampling."
        )

    eeg_picks = mne.pick_types(master.info, eeg=True, exclude=[])
    if len(eeg_picks) == 0:
        raise ValueError("No channels are typed as EEG.")

    filtered = master.copy()
    if filtered.info["sfreq"] > TARGET_SAMPLING_RATE_HZ:
        filtered.resample(
            sfreq=TARGET_SAMPLING_RATE_HZ,
            npad="auto",
            verbose="ERROR",
        )
    filtered.filter(
        l_freq=0.5,
        h_freq=50.0,
        picks=eeg_picks,
        method="fir",
        phase="zero",
        fir_design="firwin",
        skip_by_annotation=("edge", "bad_acq_skip"),
        verbose="ERROR",
    )

    # Reference with every EEG channel, including any channels already marked bad.
    original_bads = list(filtered.info["bads"])
    filtered.info["bads"] = []
    filtered.set_eeg_reference(ref_channels="average", projection=False, verbose="ERROR")
    filtered.info["bads"] = original_bads

    display_raw = filtered.copy()

    return {
        "source": source,
        "master": master,
        "filtered": filtered,
        "display": display_raw,
        "eeg_picks": list(eeg_picks),
        "original_channel_names": list(master.ch_names),
        "label_audit": [],
        "channel_audit": [],
    }


def _remove_vocab_placeholders(raw, placeholders):
    keep = []
    for index, (onset, duration, description) in enumerate(
        zip(raw.annotations.onset, raw.annotations.duration, raw.annotations.description)
    ):
        is_placeholder = (
            description in placeholders
            and abs(float(onset)) < 1e-12
            and abs(float(duration)) < 1e-12
        )
        if not is_placeholder:
            keep.append(index)
    raw.set_annotations(raw.annotations[keep] if keep else mne.Annotations([], [], []))


def sync_display_review_to_master(review):
    """Copy annotations and globally bad channels from display to all copies."""
    display_raw = review["display"]
    master = review["master"]
    filtered = review["filtered"]

    bads = list(display_raw.info["bads"])
    annotations = _copy_annotations(display_raw.annotations)
    for raw in (master, filtered):
        raw.info["bads"] = bads
        raw.set_annotations(_copy_annotations(annotations))


def open_interactive_review_viewer(review, start_sec=0.0, duration_sec=30.0):
    """Open the MNE Qt viewer and synchronize edits when it closes."""
    display_raw = review["display"]
    existing = set(str(value) for value in display_raw.annotations.description)
    placeholders = [label for label in STANDARD_LABELS if label not in existing]
    for label in placeholders:
        display_raw.annotations.append(0.0, 0.0, label)

    mne.viz.set_browser_backend("qt")
    display_raw.plot(
        start=max(0.0, float(start_sec)),
        duration=max(1.0, float(duration_sec)),
        n_channels=len(display_raw.ch_names),
        scalings={"eeg": 40e-6},
        remove_dc=False,
        block=True,
        show=True,
        title=(
            "250 Hz, 0.5–50 Hz average-referenced EEG | "
            "Click channel=bad | Press A=annotations"
        ),
    )

    _remove_vocab_placeholders(display_raw, set(placeholders))
    sync_display_review_to_master(review)


class ReviewMetadataDialog(QDialog):
    """GUI for standardized labels, channel names, and annotation insertion."""

    def __init__(self, review, parent=None):
        super().__init__(parent)
        self.review = review
        self.raw = review["master"]
        self.setWindowTitle("Standardize EEG metadata and annotations")
        self.resize(1150, 800)

        instructions = QLabel(
            "Rename channels in the table. For every annotation, choose a "
            "standard label or keep the original. Check Delete only for an "
            "incorrect annotation. New annotations may apply to all channels "
            "or selected channels."
        )
        instructions.setWordWrap(True)

        channels_box = QGroupBox("Channel labels and globally bad channels")
        channels_layout = QVBoxLayout(channels_box)
        self.channel_table = QTableWidget(len(self.raw.ch_names), 4)
        self.channel_table.setHorizontalHeaderLabels(
            ["Original channel", "New channel label", "Globally bad", "Reason"]
        )
        for row, channel in enumerate(self.raw.ch_names):
            original = QTableWidgetItem(channel)
            original.setFlags(original.flags() & ~Qt.ItemFlag.ItemIsEditable)
            self.channel_table.setItem(row, 0, original)
            self.channel_table.setItem(row, 1, QTableWidgetItem(channel))
            bad = QCheckBox()
            bad.setChecked(channel in self.raw.info["bads"])
            self.channel_table.setCellWidget(row, 2, bad)
            self.channel_table.setItem(row, 3, QTableWidgetItem(""))
        self.channel_table.resizeColumnsToContents()
        channels_layout.addWidget(self.channel_table)

        annotations_box = QGroupBox("Existing annotations")
        annotations_layout = QVBoxLayout(annotations_box)
        self.annotation_table = QTableWidget(len(self.raw.annotations), 7)
        self.annotation_table.setHorizontalHeaderLabels(
            [
                "Onset (sec)",
                "Duration (sec)",
                "Original label",
                "Standard label",
                "Channels",
                "Delete",
                "Reviewer note",
            ]
        )
        for row, (onset, duration, description, channels) in enumerate(
            zip(
                self.raw.annotations.onset,
                self.raw.annotations.duration,
                self.raw.annotations.description,
                self.raw.annotations.ch_names,
            )
        ):
            for column, value in enumerate(
                [
                    f"{float(onset):.3f}",
                    f"{float(duration):.3f}",
                    str(description),
                ]
            ):
                item = QTableWidgetItem(value)
                item.setFlags(item.flags() & ~Qt.ItemFlag.ItemIsEditable)
                self.annotation_table.setItem(row, column, item)
            label = QComboBox()
            label.addItems(["KEEP_ORIGINAL"] + STANDARD_LABELS)
            if str(description) in STANDARD_LABELS:
                label.setCurrentText(str(description))
            self.annotation_table.setCellWidget(row, 3, label)
            channel_text = ",".join(channels) if channels else "all"
            channel_item = QTableWidgetItem(channel_text)
            channel_item.setFlags(channel_item.flags() & ~Qt.ItemFlag.ItemIsEditable)
            self.annotation_table.setItem(row, 4, channel_item)
            self.annotation_table.setCellWidget(row, 5, QCheckBox())
            self.annotation_table.setItem(row, 6, QTableWidgetItem(""))
        self.annotation_table.resizeColumnsToContents()
        annotations_layout.addWidget(self.annotation_table)

        add_box = QGroupBox("Insert a new annotation")
        add_layout = QFormLayout(add_box)
        self.add_onset = QDoubleSpinBox()
        self.add_onset.setRange(0.0, float(self.raw.times[-1]))
        self.add_onset.setDecimals(3)
        self.add_duration = QDoubleSpinBox()
        self.add_duration.setRange(0.001, float(self.raw.times[-1]))
        self.add_duration.setDecimals(3)
        self.add_duration.setValue(1.0)
        self.add_label = QComboBox()
        self.add_label.addItems(STANDARD_LABELS)
        self.add_channels = QListWidget()
        self.add_channels.setSelectionMode(
            QAbstractItemView.SelectionMode.ExtendedSelection
        )
        self.add_channels.addItem("ALL")
        self.add_channels.addItems(self.raw.ch_names)
        self.add_channels.item(0).setSelected(True)
        self.add_button = QPushButton("Add annotation to table")
        self.add_button.clicked.connect(self._add_annotation_row)
        add_layout.addRow("Onset:", self.add_onset)
        add_layout.addRow("Duration:", self.add_duration)
        add_layout.addRow("Standard label:", self.add_label)
        add_layout.addRow("Affected channels:", self.add_channels)
        add_layout.addRow("", self.add_button)

        zoom_box = QGroupBox("Optional range to reopen in the EEG viewer")
        zoom_layout = QFormLayout(zoom_box)
        self.zoom_start = QDoubleSpinBox()
        self.zoom_start.setRange(0.0, float(self.raw.times[-1]))
        self.zoom_start.setDecimals(3)
        self.zoom_end = QDoubleSpinBox()
        self.zoom_end.setRange(0.001, float(self.raw.times[-1]))
        self.zoom_end.setDecimals(3)
        self.zoom_end.setValue(min(120.0, float(self.raw.times[-1])))
        zoom_layout.addRow("Start:", self.zoom_start)
        zoom_layout.addRow("End:", self.zoom_end)

        buttons = QDialogButtonBox(
            QDialogButtonBox.StandardButton.Apply
            | QDialogButtonBox.StandardButton.Cancel
        )
        buttons.button(QDialogButtonBox.StandardButton.Apply).clicked.connect(
            self._validate_and_accept
        )
        buttons.rejected.connect(self.reject)

        bottom = QHBoxLayout()
        bottom.addWidget(add_box, 2)
        bottom.addWidget(zoom_box, 1)

        layout = QVBoxLayout(self)
        layout.addWidget(instructions)
        layout.addWidget(channels_box, 1)
        layout.addWidget(annotations_box, 2)
        layout.addLayout(bottom)
        layout.addWidget(buttons)

    def _add_annotation_row(self):
        selected = [item.text() for item in self.add_channels.selectedItems()]
        if not selected:
            QMessageBox.warning(self, "Missing channels", "Select ALL or channels.")
            return
        onset = float(self.add_onset.value())
        duration = float(self.add_duration.value())
        if onset + duration > self.raw.times[-1] + 1 / self.raw.info["sfreq"]:
            QMessageBox.warning(
                self, "Invalid interval", "The interval extends past the recording."
            )
            return

        row = self.annotation_table.rowCount()
        self.annotation_table.insertRow(row)
        values = [
            f"{onset:.3f}",
            f"{duration:.3f}",
            "(new annotation)",
        ]
        for column, value in enumerate(values):
            item = QTableWidgetItem(value)
            item.setFlags(item.flags() & ~Qt.ItemFlag.ItemIsEditable)
            self.annotation_table.setItem(row, column, item)
        label = QComboBox()
        label.addItems(STANDARD_LABELS)
        label.setCurrentText(self.add_label.currentText())
        self.annotation_table.setCellWidget(row, 3, label)
        channel_text = "all" if "ALL" in selected else ",".join(selected)
        channel_item = QTableWidgetItem(channel_text)
        channel_item.setFlags(channel_item.flags() & ~Qt.ItemFlag.ItemIsEditable)
        self.annotation_table.setItem(row, 4, channel_item)
        self.annotation_table.setCellWidget(row, 5, QCheckBox())
        self.annotation_table.setItem(row, 6, QTableWidgetItem("manually inserted"))

    def _validate_and_accept(self):
        new_names = [
            self.channel_table.item(row, 1).text().strip()
            for row in range(self.channel_table.rowCount())
        ]
        if any(not value for value in new_names):
            QMessageBox.warning(self, "Invalid channel label", "Channel labels cannot be blank.")
            return
        if len(set(new_names)) != len(new_names):
            QMessageBox.warning(self, "Duplicate channel label", "Channel labels must be unique.")
            return
        if self.zoom_end.value() <= self.zoom_start.value():
            QMessageBox.warning(self, "Invalid range", "Zoom end must be after zoom start.")
            return
        self.accept()

    def apply_changes(self):
        old_names = list(self.raw.ch_names)
        new_names = [
            self.channel_table.item(row, 1).text().strip()
            for row in range(self.channel_table.rowCount())
        ]
        rename_map = {
            old: new for old, new in zip(old_names, new_names) if old != new
        }
        bad_old_names = [
            old_names[row]
            for row in range(len(old_names))
            if self.channel_table.cellWidget(row, 2).isChecked()
        ]
        reasons = {
            old_names[row]: self.channel_table.item(row, 3).text().strip()
            for row in range(len(old_names))
            if self.channel_table.item(row, 3).text().strip()
        }

        onset = []
        duration = []
        description = []
        ch_names = []
        label_audit = []
        for row in range(self.annotation_table.rowCount()):
            if self.annotation_table.cellWidget(row, 5).isChecked():
                continue
            row_onset = float(self.annotation_table.item(row, 0).text())
            row_duration = float(self.annotation_table.item(row, 1).text())
            raw_label = self.annotation_table.item(row, 2).text()
            selected_label = self.annotation_table.cellWidget(row, 3).currentText()
            standard_label = (
                raw_label if selected_label == "KEEP_ORIGINAL" else selected_label
            )
            channel_text = self.annotation_table.item(row, 4).text()
            old_channels = [] if channel_text == "all" else channel_text.split(",")
            mapped_channels = [rename_map.get(value, value) for value in old_channels]
            note = self.annotation_table.item(row, 6).text().strip()

            onset.append(row_onset)
            duration.append(row_duration)
            description.append(standard_label)
            ch_names.append(mapped_channels)
            label_audit.append(
                {
                    "onset_sec": row_onset,
                    "duration_sec": row_duration,
                    "raw_label": raw_label,
                    "standard_label": standard_label,
                    "reviewer_note": note,
                }
            )

        for raw in (
            self.review["master"],
            self.review["filtered"],
            self.review["display"],
        ):
            if rename_map:
                raw.rename_channels(rename_map)
            raw.info["bads"] = [rename_map.get(name, name) for name in bad_old_names]
            raw.set_annotations(
                mne.Annotations(
                    onset=onset,
                    duration=duration,
                    description=description,
                    orig_time=raw.annotations.orig_time,
                    ch_names=ch_names,
                )
            )

        self.review["label_audit"] = label_audit
        self.review["channel_audit"] = [
            {
                "original_channel": old,
                "standard_channel": rename_map.get(old, old),
                "globally_bad": old in bad_old_names,
                "reason": reasons.get(old, ""),
            }
            for old in old_names
        ]
        return float(self.zoom_start.value()), float(self.zoom_end.value())


def open_metadata_review_gui(review):
    _qt_app()
    dialog = ReviewMetadataDialog(review)
    if dialog.exec() != QDialog.DialogCode.Accepted:
        raise RuntimeError("Metadata review was cancelled; nothing was changed.")
    return dialog.apply_changes()


def sync_legacy_eeg_dictionary(review, eeg):
    """Keep the notebook's later Plotly/spectrogram cells compatible."""
    raw = review["master"]
    eeg["labels"] = list(raw.ch_names)
    eeg["markers"] = [
        {
            "label": str(description),
            "sample": int(round(float(onset) * raw.info["sfreq"])) + 1,
            "time_sec": float(onset),
            "duration_sec": float(duration),
            "channels": list(channels),
        }
        for onset, duration, description, channels in zip(
            raw.annotations.onset,
            raw.annotations.duration,
            raw.annotations.description,
            raw.annotations.ch_names,
        )
    ]
    return eeg


class SaveReviewDialog(QDialog):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.setWindowTitle("Save reviewed EEG")
        self.resize(520, 330)

        explanation = QLabel(
            "Use deidentified identifiers only. The source BrainVision files "
            "will not be changed. A new versioned folder will be created."
        )
        explanation.setWordWrap(True)

        self.participant = QLineEdit("L000")
        self.run = QLineEdit("run-01")
        self.reviewer = QLineEdit()
        self.context = QComboBox()
        self.context.addItems(
            ["preop1", "preop2", "or1", "or2", "or3", "or4"]
        )

        form = QFormLayout()
        form.addRow("Participant:", self.participant)
        form.addRow("Recording:", self.run)
        form.addRow("Reviewer initials/ID:", self.reviewer)
        form.addRow("Processing sections:", QLabel("Defined by the Step 4 selections"))

        buttons = QDialogButtonBox(
            QDialogButtonBox.StandardButton.Save
            | QDialogButtonBox.StandardButton.Cancel
        )
        buttons.button(QDialogButtonBox.StandardButton.Save).clicked.connect(
            self._validate_and_accept
        )
        buttons.rejected.connect(self.reject)

        layout = QVBoxLayout(self)
        layout.addWidget(explanation)
        layout.addLayout(form)
        layout.addWidget(buttons)

    def _validate_and_accept(self):
        participant = self.participant.text().strip()
        if not re.fullmatch(r"L[0-9]{3,}", participant, flags=re.IGNORECASE):
            QMessageBox.warning(
                self,
                "Invalid participant",
                "Use a deidentified participant ID such as L000 or L0123.",
            )
            return
        run = self.run.text().strip()
        if not re.fullmatch(r"run-[A-Za-z0-9]+", run):
            QMessageBox.warning(
                self,
                "Invalid recording",
                "Use a recording value such as run-01.",
            )
            return
        if not re.fullmatch(r"[A-Za-z0-9_-]{1,20}", self.reviewer.text().strip()):
            QMessageBox.warning(
                self,
                "Invalid reviewer",
                "Enter 1–20 initials/ID characters without spaces.",
            )
            return
        self.accept()

    def values(self):
        return {
            "participant": self.participant.text().strip(),
            "run": self.run.text().strip(),
            "reviewer": self.reviewer.text().strip(),
            "context": self.context.currentText(),
        }


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _write_tsv(path, rows, columns):
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, delimiter="\t")
        writer.writeheader()
        for row in rows:
            writer.writerow({column: row.get(column, "") for column in columns})


def save_review_gui(review, export_folder):
    """Collect deidentified metadata and save a versioned review bundle."""
    _qt_app()
    dialog = SaveReviewDialog()
    if dialog.exec() != QDialog.DialogCode.Accepted:
        raise RuntimeError("Save was cancelled.")
    values = dialog.values()

    participant = values["participant"]
    run = values["run"]
    reviewer = values["reviewer"]
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    context = values["context"]
    base = f"{participant}_task-anesthesia_acq-{context}_{run}"
    output = (
        Path(export_folder)
        / participant
        / "eeg"
        / f"review-{stamp}"
    )
    output.mkdir(parents=True, exist_ok=False)

    master = review["master"]
    processed = review["filtered"].copy().load_data()
    processed.set_annotations(master.annotations.copy())
    processed.info["bads"] = list(master.info["bads"])

    reviewed_path = output / f"{base}_desc-filteredAverageRef_raw.fif"
    processed.save(reviewed_path, overwrite=False, verbose="ERROR")

    annotation_rows = []
    for onset, duration, description, channels in zip(
        master.annotations.onset,
        master.annotations.duration,
        master.annotations.description,
        master.annotations.ch_names,
    ):
        annotation_rows.append(
            {
                "onset_sec": float(onset),
                "duration_sec": float(duration),
                "standard_label": str(description),
                "channels": ",".join(channels) if channels else "all",
                "reviewer": reviewer,
                "review_version": stamp,
            }
        )
    _write_tsv(
        output / f"{base}_annotations.tsv",
        annotation_rows,
        [
            "onset_sec",
            "duration_sec",
            "standard_label",
            "channels",
            "reviewer",
            "review_version",
        ],
    )
    _write_tsv(
        output / f"{base}_label_audit.tsv",
        review["label_audit"],
        [
            "onset_sec",
            "duration_sec",
            "raw_label",
            "standard_label",
            "reviewer_note",
        ],
    )
    _write_tsv(
        output / f"{base}_channels.tsv",
        review["channel_audit"],
        [
            "original_channel",
            "standard_channel",
            "globally_bad",
            "reason",
        ],
    )

    summary = {
        "participant_id": participant,
        "run_id": run,
        "reviewer": reviewer,
        "file_context": values["context"],
        "review_created_utc": stamp,
        "source_extension": review["source"].suffix.lower(),
        "source_sha256": _sha256(review["source"]),
        "source_filename_omitted_to_reduce_phi_risk": True,
        "source_sampling_frequency_hz": float(master.info["sfreq"]),
        "sampling_frequency_hz": float(processed.info["sfreq"]),
        "n_channels": len(master.ch_names),
        "globally_bad_channels": list(master.info["bads"]),
        "n_annotations": len(master.annotations),
        "saved_signal": {
            "bandpass_hz": [0.5, 50.0],
            "resampled_to_hz": float(processed.info["sfreq"]),
            "reference": "common average across all EEG channels",
            "saved_as_authoritative_signal": True,
            "normalization": "one global z-score across all EEG channels and samples after common average reference",
            "data_unit": "global z-score (dimensionless)",
            "power_spectral_density_unit": "global z-score squared per Hz",
        },
        "ica_applied": False,
        "physical_unit_eeg_saved": False,
        "average_referenced": True,
        "global_zscore_saved": True,
    }
    with (output / f"{base}_review.json").open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)

    QMessageBox.information(
        None,
        "Review saved",
        "The reviewed 250 Hz, 0.5–50 Hz filtered, average-referenced global z-score EEG "
        "and audit tables were saved to:\n\n"
        f"{output}\n\n"
        "Reopen the reviewed FIF for verification before sharing.",
    )
    return output


def make_plotly_display_eeg(review, eeg):
    """Return the legacy EEG dictionary using the filtered physical signal."""
    eeg = sync_legacy_eeg_dictionary(review, eeg)
    display_eeg = dict(eeg)
    display_eeg["data"] = review["filtered"].get_data()
    display_eeg["labels"] = list(review["filtered"].ch_names)
    display_eeg["srate"] = float(review["filtered"].info["sfreq"])
    display_eeg["n_samples"] = int(review["filtered"].n_times)
    display_eeg["duration_sec"] = (
        float(review["filtered"].times[-1])
        if review["filtered"].n_times
        else 0.0
    )
    display_eeg["markers"] = list(eeg["markers"])
    return display_eeg


## 3. choose and load one queued recording.

In [6]:
# Step 3: choose and load one queued recording.
# A Windows dialog asks which selected file to review.
# This creates:
#   review['master']   = original physical-unit EEG with editable metadata
#   review['filtered'] = separate 250 Hz, 0.5–50 Hz average-referenced global z-score copy
#   review['display']  = GUI-scaled copy of the global z-score data

review = prepare_gui_recording(VHDR_FILES)
VHDR_FILE = str(review["source"])

# Keep the original notebook dictionary for the later spectrogram/export cells.
eeg = load_brainvision(VHDR_FILE)

print("Loaded:", Path(VHDR_FILE).name)
print("Channels:", len(review["master"].ch_names), review["master"].ch_names)
print("Source sampling rate:", review["master"].info["sfreq"], "Hz")
print("Processed sampling rate:", review["filtered"].info["sfreq"], "Hz")
print("Duration:", round(review["master"].times[-1], 3), "seconds")
print("Existing annotations:", len(review["master"].annotations))
print("Ready for Step 4.")


Loaded: 20260805OR.vhdr
Channels: 8 ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']
Source sampling rate: 1000.0 Hz
Processed sampling rate: 250.0 Hz
Duration: 7056.079 seconds
Existing annotations: 11
Ready for Step 4.


## 4. interactive plotly.

In [ ]:
# Step 4: interactive Plotly/Dash EEG annotation interface.
#
# Processing order:
#   original EEG → 250 Hz resampling → 0.5–50 Hz filtering → common average reference
#
# This interface allows:
#   • box-selection of artifact intervals;
#   • clicking to select point events;
#   • automatic conversion from plot minutes to exact seconds;
#   • standardized annotation labels;
#   • ALL-channel or channel-specific annotations;
#   • undoing newly added annotations.
#
# Step 4 displays the physical-unit, average-referenced signal; global z-scoring happens only after section selection.

import importlib.util
import socket
import subprocess
import sys
import threading
import webbrowser
from datetime import datetime, timezone
from pathlib import Path
import json

import h5py

import numpy as np
import plotly.graph_objects as go

# Install Dash if it is not available.
if importlib.util.find_spec("dash") is None:

    print("Installing Dash...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "dash",
        ]
    )

from dash import (
    Dash,
    Input,
    Output,
    State,
    ctx,
    dash_table,
    dcc,
    html,
    no_update,
)
from werkzeug.serving import make_server

# ---------------------------------------------------------
# Prepare the filtered EEG.
# ---------------------------------------------------------

eeg_display = make_plotly_display_eeg(
    review,
    eeg,
)

data_filtered = eeg_display["data"]

channel_labels = list(
    eeg_display["labels"]
)

sampling_rate = float(
    eeg_display["srate"]
)

number_samples = int(
    eeg_display["n_samples"]
)

print(
    "Display filtering:",
    review["filtered"].info["highpass"],
    "to",
    review["filtered"].info["lowpass"],
    "Hz",
)

# High-variance candidates are detected on demand in the webpage.
_initial_display = data_filtered[:, ::max(1, int(round(sampling_rate)))].astype(float, copy=False)
_initial_robust_sd = 1.4826 * float(np.nanmedian(np.abs(_initial_display - np.nanmedian(_initial_display))))
HIGH_VARIANCE_CURRENT_ROBUST_SD = _initial_robust_sd


# ---------------------------------------------------------
# Downsample the average-referenced signal for browser visualization.
# ---------------------------------------------------------

MAX_DISPLAY_POINTS = 20000

display_step = max(
    1,
    int(
        np.ceil(
            number_samples
            / MAX_DISPLAY_POINTS
        )
    ),
)

display_samples = np.arange(
    0,
    number_samples,
    display_step,
)

time_minutes = (
    display_samples
    / sampling_rate
    / 60.0
)

data_to_plot = data_filtered[:, display_samples].astype(np.float32, copy=False) * 1e6

# Use a robust display limit so isolated artifacts do not make the EEG look flat.
DISPLAY_AMPLITUDE_LIMIT_UV = float(np.nanpercentile(np.abs(data_to_plot), 98.0))
if not np.isfinite(DISPLAY_AMPLITUDE_LIMIT_UV) or DISPLAY_AMPLITUDE_LIMIT_UV == 0:
    DISPLAY_AMPLITUDE_LIMIT_UV = 1.0

data_to_plot = np.clip(
    data_to_plot,
    -DISPLAY_AMPLITUDE_LIMIT_UV,
    DISPLAY_AMPLITUDE_LIMIT_UV,
)
# Keep the physical average-referenced display so z-score preview can be undone.
PHYSICAL_DATA_TO_PLOT = data_to_plot.copy()
STEP4_DISPLAY_MODE = "average-referenced µV"

OVERVIEW_DISPLAY_SAMPLES = display_samples.copy()
OVERVIEW_DISPLAY_AMPLITUDE_LIMIT_UV = DISPLAY_AMPLITUDE_LIMIT_UV
ADAPTIVE_FULL_RES_MAX_SECONDS = 180.0
STEP4_HIGH_RES_ACTIVE = False

def set_adaptive_display_window(start_minute=None, end_minute=None):
    """Use 250 Hz in a short focused interval; retain the overview otherwise."""
    global display_samples, time_minutes, data_to_plot, DISPLAY_AMPLITUDE_LIMIT_UV, CHANNEL_SPACING, offsets
    if start_minute is None or end_minute is None:
        display_samples = OVERVIEW_DISPLAY_SAMPLES.copy()
    else:
        start = max(0, int(np.floor(float(start_minute) * 60.0 * sampling_rate)))
        stop = min(number_samples, int(np.ceil(float(end_minute) * 60.0 * sampling_rate)))
        if stop <= start:
            return
        span = stop - start
        maximum = int(ADAPTIVE_FULL_RES_MAX_SECONDS * sampling_rate)
        step = max(1, int(np.ceil(span / maximum)))
        display_samples = np.arange(start, stop, step, dtype=int)
    time_minutes = display_samples / sampling_rate / 60.0
    if STEP4_DISPLAY_MODE == "global z-score":
        return
    data_to_plot = data_filtered[:, display_samples].astype(np.float32, copy=False) * 1e6
    if start_minute is None or end_minute is None:
        DISPLAY_AMPLITUDE_LIMIT_UV = OVERVIEW_DISPLAY_AMPLITUDE_LIMIT_UV
    else:
        focused_limit = float(np.nanpercentile(np.abs(data_to_plot), 98.0))
        if np.isfinite(focused_limit) and focused_limit > 0:
            DISPLAY_AMPLITUDE_LIMIT_UV = focused_limit
    data_to_plot = np.clip(data_to_plot, -DISPLAY_AMPLITUDE_LIMIT_UV, DISPLAY_AMPLITUDE_LIMIT_UV)
    CHANNEL_SPACING = 2.5 * DISPLAY_AMPLITUDE_LIMIT_UV
    offsets = np.arange(number_channels)[::-1] * CHANNEL_SPACING

# Compact channel separation keeps the complete channel stack on one screen.
CHANNEL_SPACING = 2.5 * DISPLAY_AMPLITUDE_LIMIT_UV

number_channels = len(
    channel_labels
)

offsets = (
    np.arange(number_channels)[::-1]
    * CHANNEL_SPACING
)

# ---------------------------------------------------------
# High-contrast channel colors.
# ---------------------------------------------------------

CHANNEL_COLORS = {
    "FP1": "#0072B2",
    "FP2": "#D55E00",
    "F7": "#009E73",
    "F8": "#CC79A7",
    "T7": "#E69F00",
    "T8": "#56B4E9",
    "O1": "#222222",
    "O2": "#7B2CBF",
}

FALLBACK_COLORS = [
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#CC79A7",
    "#E69F00",
    "#56B4E9",
    "#222222",
    "#7B2CBF",
]

top_of_plot = (
    offsets[0]
    + DISPLAY_AMPLITUDE_LIMIT_UV
)

bottom_of_plot = (
    offsets[-1]
    - DISPLAY_AMPLITUDE_LIMIT_UV
)

MARKER_LABEL_Y = (
    top_of_plot + 0.65 * DISPLAY_AMPLITUDE_LIMIT_UV
)
MAX_INTERVAL_OVERLAYS = 250
MAX_VISIBLE_MARKER_LABELS = 80

# Newly created Step 4 annotations are held here
# until Step 5 transfers them into the reviewed EEG.
STEP4_MANUAL_ANNOTATIONS = []

# Number already transferred to MNE.
STEP4_SYNCED_COUNT = 0
STEP4_DISPLAY_GAIN = 1.0
STEP4_HIGH_VARIANCE_CANDIDATES = []
STEP4_RECORDING_SCALE = None
STEP4_VIEW_START_MINUTE = None
STEP4_VIEW_END_MINUTE = None
# The focused interval is the only data Step 5 is allowed to process.
STEP4_FOCUSED_INTERVAL = None
# Exactly one interval can be assigned a context for the next Step 5 export.
STEP4_PROCESSING_SEGMENTS = []


def build_interactive_eeg_figure(display_gain=None):
    """Build the EEG figure with existing and newly added annotations."""

    if display_gain is None:
        display_gain = STEP4_DISPLAY_GAIN
    display_values = data_to_plot * float(display_gain)
    hover_value_label = "Global z-score" if STEP4_DISPLAY_MODE == "global z-score" else "Amplitude (µV)"
    display_top = offsets[0] + DISPLAY_AMPLITUDE_LIMIT_UV * float(display_gain)
    display_bottom = offsets[-1] - DISPLAY_AMPLITUDE_LIMIT_UV * float(display_gain)
    # Keep annotation labels visible above the traces at every display gain.
    display_marker_y = offsets[0] + 1.45 * DISPLAY_AMPLITUDE_LIMIT_UV

    figure = go.Figure()

    # Add EEG channels.
    for channel_index, channel_label in enumerate(
        channel_labels
    ):

        normalized_label = (
            channel_label
            .strip()
            .upper()
            .replace("EEG ", "")
        )

        channel_color = CHANNEL_COLORS.get(
            normalized_label,
            FALLBACK_COLORS[
                channel_index
                % len(FALLBACK_COLORS)
            ],
        )

        figure.add_trace(
            go.Scattergl(
                x=time_minutes,
                y=(
                    display_values[
                        channel_index
                    ]
                    + offsets[
                        channel_index
                    ]
                ),
                mode="lines",
                name=channel_label,
                line={
                    "width": 1.5,
                    "color": channel_color,
                },
                customdata=data_to_plot[
                    channel_index
                ],
                hovertemplate=(
                    f"<b>{channel_label}</b><br>"
                    "Time: %{x:.4f} min<br>"
                    f"{hover_value_label}: %{{customdata:.2f}}"
                    "<extra></extra>"
                ),
            )
        )

    # Combine imported annotations and new annotations.
    all_markers = []

    for marker in eeg_display["markers"]:

        all_markers.append(
            {
                "time_sec": float(
                    marker["time_sec"]
                ),
                "duration_sec": float(
                    marker.get(
                        "duration_sec",
                        0.0,
                    )
                ),
                "label": str(
                    marker["label"]
                ),
                "channels": marker.get(
                    "channels",
                    ["ALL"],
                ),
                "is_manual": False,
            }
        )

    all_markers.extend({**marker, "is_manual": True} for marker in STEP4_MANUAL_ANNOTATIONS)
    all_markers.extend({**marker, "is_manual": True} for marker in STEP4_HIGH_VARIANCE_CANDIDATES)

    def visible_subset(markers, maximum):
        """Keep labels representative and responsive for dense event streams."""
        manual = [marker for marker in markers if marker.get("is_manual")]
        imported = [marker for marker in markers if not marker.get("is_manual")]
        remaining = max(0, maximum - len(manual))
        if len(imported) <= remaining:
            return manual + imported
        selected_indices = np.linspace(0, len(imported) - 1, remaining, dtype=int)
        return manual + [imported[index] for index in selected_indices]

    # Render all point annotations in one WebGL trace. This avoids creating
    # thousands of Plotly shapes and labels for dense imported event streams.
    point_markers = [
        marker for marker in all_markers
        if float(marker.get("duration_sec", 0.0)) <= 0
    ]
    if point_markers:
        figure.add_trace(
            go.Scattergl(
                x=[float(marker["time_sec"]) / 60.0 for marker in point_markers],
                y=[display_marker_y] * len(point_markers),
                mode="markers",
                marker={"size": 6, "color": "red", "symbol": "line-ns-open"},
                customdata=[
                    f"{marker['label']} [{','.join(marker.get('channels', ['ALL']))}]"
                    for marker in point_markers
                ],
                hovertemplate=(
                    "Point annotation<br>Time: %{x:.4f} min<br>%{customdata}"
                    "<extra></extra>"
                ),
            )
        )

    # Labels are always visible for manually added markers. Imported labels are
    # evenly sampled when there are many, while every marker remains hoverable.
    label_markers = visible_subset(all_markers, MAX_VISIBLE_MARKER_LABELS)
    if label_markers:
        figure.add_trace(
            go.Scatter(
                x=[float(marker["time_sec"]) / 60.0 for marker in label_markers],
                y=[display_marker_y] * len(label_markers),
                mode="markers+text",
                marker={"size": 7, "color": "crimson"},
                text=[
                    f"{marker['label']} [{','.join(marker.get('channels', []))}]"
                    if str(marker["label"]) == "BAD_channel" and marker.get("channels")
                    else str(marker["label"])
                    for marker in label_markers
                ],
                textposition="top center",
                textfont={"color": "crimson", "size": 10},
                hovertemplate=(
                    "Annotation: %{text}<br>Time: %{x:.4f} min"
                    "<extra></extra>"
                ),
                cliponaxis=False,
            )
        )

    # Interval annotations retain their shaded spans and labels.
    interval_markers = [
        marker for marker in all_markers
        if float(marker.get("duration_sec", 0.0)) > 0
    ]
    for marker in visible_subset(interval_markers, MAX_INTERVAL_OVERLAYS):

        marker_time_sec = float(
            marker["time_sec"]
        )

        marker_duration_sec = float(
            marker.get(
                "duration_sec",
                0.0,
            )
        )

        marker_time_min = (
            marker_time_sec / 60.0
        )

        marker_label = str(
            marker["label"]
        )

        if marker_duration_sec <= 0:
            continue

        if marker_duration_sec > 0:

            marker_end_min = (
                marker_time_sec
                + marker_duration_sec
            ) / 60.0

            if marker_label.startswith(
                "BAD_"
            ):

                fill_color = (
                    "rgba(255, 0, 0, 0.15)"
                )

            elif marker_label.startswith(
                "CLEAN_"
            ):

                fill_color = (
                    "rgba(0, 170, 0, 0.10)"
                )

            else:

                fill_color = (
                    "rgba(30, 100, 255, 0.10)"
                )

            marker_channels = list(marker.get("channels", []))
            if marker_label == "BAD_channel" and marker_channels and "ALL" not in marker_channels:
                # Channel-specific bad marking: shade only the affected rows,
                # rather than implying that every channel is bad.
                for affected_channel in marker_channels:
                    if affected_channel not in channel_labels:
                        continue
                    channel_index = channel_labels.index(affected_channel)
                    figure.add_shape(
                        type="rect", xref="x", yref="y",
                        x0=marker_time_min, x1=marker_end_min,
                        y0=offsets[channel_index] - 0.42 * CHANNEL_SPACING,
                        y1=offsets[channel_index] + 0.42 * CHANNEL_SPACING,
                        fillcolor=fill_color, line_width=0, layer="below",
                    )
            else:
                figure.add_vrect(
                    x0=marker_time_min,
                    x1=marker_end_min,
                    fillcolor=fill_color,
                    line_width=0,
                    layer="below",
                )

        figure.add_vline(
            x=marker_time_min,
            line_color="red",
            line_dash="dash",
            line_width=1,
            opacity=0.65,
        )

    figure.update_layout(
        title=(
            "EEG review — " + STEP4_DISPLAY_MODE + ": drag a box for an interval "
            "or click for a point event"
        ),
        xaxis_title=(
            "Time from beginning of recording "
            "(minutes)"
        ),
        yaxis={
            "title": "Channels (global z-score)" if STEP4_DISPLAY_MODE == "global z-score" else "Channels (average-referenced µV)",
            "tickmode": "array",
            "tickvals": offsets,
            "ticktext": channel_labels,
            # Keep channel baseline positions fixed as display gain changes.
        "range": [
                offsets[-1] - 1.25 * DISPLAY_AMPLITUDE_LIMIT_UV,
                offsets[0] + 2.25 * DISPLAY_AMPLITUDE_LIMIT_UV,
            ],
        },
        height=max(630, 60 * number_channels),
        hovermode="closest",
        showlegend=False,
        dragmode="select",
        clickmode="event+select",
        uirevision="keep-eeg-view",
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin={
            "l": 110,
            "r": 30,
            "t": 85,
            "b": 60,
        },
    )

    figure.update_xaxes(
        title="Time from recording start (minutes; dashed minor grid = 1 second)",
        showgrid=True,
        gridcolor="rgba(180, 180, 180, 0.25)",
        griddash="solid",
        showticklabels=True,
        ticks="outside",
        rangeslider={"visible": True, "thickness": 0.06},
        range=(
            [STEP4_VIEW_START_MINUTE, STEP4_VIEW_END_MINUTE]
            if STEP4_VIEW_START_MINUTE is not None
            else None
        ),
        minor={
            "dtick": 1.0 / 60.0,
            "showgrid": True,
            "gridcolor": "rgba(110, 110, 110, 0.38)",
            "griddash": "dash",
        },
    )

    figure.update_yaxes(
        showgrid=False,
    )

    return figure


# ---------------------------------------------------------
# Create the Dash application.
# ---------------------------------------------------------

step4_app = Dash(
    __name__,
)

channel_options = [
    {
        "label": "ALL channels",
        "value": "ALL",
    }
]

channel_options.extend(
    {
        "label": channel,
        "value": channel,
    }
    for channel in channel_labels
)

label_options = [{"label": label, "value": label} for label in STANDARD_LABELS]
existing_annotation_options = [
    {"label": f"{float(onset):.3f} sec — {description}", "value": str(index)}
    for index, (onset, description) in enumerate(zip(review["master"].annotations.onset, review["master"].annotations.description))
]

PROCESSING_CONTEXTS = ["preop1", "preop2", "or1", "or2", "or3", "or4", "or5"]
ALLOWED_CHANNEL_NAMES = ["Fp1", "Fp2", "F3", "F4", "FC1", "FC2", "C3", "C4", "Fz", "Cz", "P3", "P", "O1", "O2"]

step4_app.layout = html.Div([
    html.Div([
        html.Div([
html.H3("Recording-wide normalization"),
            html.Label(f"Exclude samples above robust z-score threshold when fitting (current robust SD = {HIGH_VARIANCE_CURRENT_ROBUST_SD:.3e})"),
            dcc.Input(id="high-variance-threshold", type="number", value=3.0, min=0.5, step=0.5, style={"width":"100%"}),
            html.Div("This controls the scale fit only; it does not create annotations or clip the saved EEG.", style={"fontSize":"11px", "marginTop":"4px"}),
            html.Button("Preview global z-score", id="preview-zscore", n_clicks=0, style={"marginTop":"6px"}), html.Button("Show average reference", id="show-average-reference", n_clicks=0, style={"marginLeft":"6px"}), html.Div(id="zscore-preview-status", style={"marginTop":"4px","fontSize":"11px"}),
            html.Hr(), html.H3("Focused interval"),
            html.Div([html.Button("Focus selected interval", id="focus-selected-interval", n_clicks=0), html.Button("Show full recording", id="show-full-recording", n_clicks=0, style={"marginLeft": "6px"}), html.Button("Clear focus selection", id="clear-focus-selection", n_clicks=0, style={"marginLeft": "6px"})]),
            html.Div([dcc.Input(id="start-minutes", type="number", step=0.0001, placeholder="Start min", style={"width": "48%"}), dcc.Input(id="end-minutes", type="number", step=0.0001, placeholder="End min", style={"width": "48%", "marginLeft": "4%"})], style={"marginTop": "8px"}),
            html.Div(id="selection-status", children="Drag an interval in the EEG plot, then focus it.", style={"marginTop": "8px", "fontSize": "12px"}),
            html.Hr(), html.Label("Display gain"),
            html.Div([html.Button("− ×0.5", id="decrease-display-gain", n_clicks=0), html.Button("+ ×2", id="increase-display-gain", n_clicks=0, style={"marginLeft": "6px"}), html.Button("Reset gain / full view", id="reset-display-gain", n_clicks=0, style={"marginLeft": "6px"}), html.Button("Switch to 250 Hz view", id="toggle-full-rate-view", n_clicks=0, style={"marginTop": "6px"})]),
            html.Div(id="action-status", style={"marginTop": "8px", "fontWeight": "bold", "fontSize": "12px"}),
        ], style={"width": "300px", "padding": "10px", "border": "1px solid #ccc", "borderRadius": "6px", "backgroundColor": "#fafafa"}),
        html.Div([dcc.Graph(id="eeg-graph", figure=build_interactive_eeg_figure(), config={"displaylogo": False, "scrollZoom": True, "doubleClick": "reset+autosize", "responsive": True, "modeBarButtonsToRemove": ["lasso2d"]}, style={"height": f"{max(630, 60 * number_channels)}px"})], style={"flex": "1", "minWidth": "0"}),
    ], style={"display": "flex", "gap": "10px", "alignItems": "stretch"}),
    html.Div([
        html.Div([
            html.H4("Bad channels"),
            html.Div("Set global bad channels before previewing or packaging. Choose focused interval only after focusing an interval.", style={"fontSize": "11px", "marginBottom": "5px"}),
            dcc.RadioItems(id="bad-channel-scope", options=[{"label": "Global", "value": "global"}, {"label": "Focused interval only", "value": "focused"}], value="global"),
            dcc.Dropdown(id="bad-channel-selection", options=[{"label": value, "value": value} for value in channel_labels], value=[], multi=True, placeholder="Select channels", style={"marginTop": "6px"}),
            html.Div([html.Button("Set bad", id="set-bad-channels", n_clicks=0), html.Button("Clear global", id="clear-bad-channels", n_clicks=0, style={"marginLeft": "6px"})], style={"marginTop": "6px"}),
            html.Div(id="bad-channel-status", style={"fontSize": "11px", "marginTop": "6px"}),
        ]),
        html.Div([
            html.H4("Add annotation"),
            dcc.RadioItems(id="annotation-kind", options=[{"label": "Interval", "value": "interval"}, {"label": "Point", "value": "point"}], value="interval", inline=True),
            dcc.Dropdown(id="standard-label", options=label_options, value="BAD_movement", clearable=False, style={"marginTop": "6px"}),
            dcc.Dropdown(id="affected-channels", options=channel_options, value=["ALL"], multi=True, placeholder="Affected channels", style={"marginTop": "6px"}),
            html.Div([html.Button("Add", id="add-annotation", n_clicks=0), html.Button("Undo", id="undo-annotation", n_clicks=0, style={"marginLeft": "6px"})], style={"marginTop": "6px"}),
        ]),
        html.Div([
            html.H4("Standardize / rename"),
            dcc.Dropdown(id="replace-annotation-index", options=existing_annotation_options, placeholder="Existing annotation"),
            dcc.Dropdown(id="replace-annotation-label", options=label_options, value="BAD_movement", clearable=False, style={"marginTop": "6px"}),
            html.Button("Replace label", id="replace-annotation", n_clicks=0, style={"marginTop": "6px"}), html.Button("Delete label", id="delete-annotation", n_clicks=0, style={"marginLeft": "6px"}), html.Div(id="replace-annotation-status", style={"fontSize": "11px"}),
            dcc.Input(id="batch-delete-search", type="text", placeholder="Search text (blank = duplicate-label suggestions)", style={"width":"100%","marginTop":"6px"}),
            html.Button("Find / suggest", id="batch-delete-find", n_clicks=0), html.Button("Delete selected", id="batch-delete-confirm", n_clicks=0, style={"marginLeft":"6px"}),
            dcc.Checklist(id="batch-delete-selection", options=[], value=[], style={"fontSize":"11px", "maxHeight":"82px", "overflowY":"auto", "marginTop":"4px"}),
            dcc.Checklist(id="batch-delete-ok", options=[{"label":"I confirm batch deletion","value":"yes"}], value=[], style={"fontSize":"11px"}), html.Div(id="batch-delete-status", children="Search text, or click Find / suggest with blank text to list duplicate labels.", style={"fontSize":"11px"}),
            dcc.Dropdown(id="channel-rename-source", options=[{"label": value, "value": value} for value in channel_labels], value=channel_labels[0] if channel_labels else None, clearable=False, style={"marginTop": "8px"}),
            dcc.Dropdown(id="channel-rename-target", options=[{"label": value, "value": value} for value in ALLOWED_CHANNEL_NAMES], value="Fp1", clearable=False, style={"marginTop": "6px"}),
            html.Button("Rename channel", id="apply-channel-rename", n_clicks=0, style={"marginTop": "6px"}), html.Div(id="channel-rename-status", style={"fontSize": "11px"}),
        ]),
        html.Div([
            html.H4("Package interval (focused, or whole recording)"),
            dcc.Input(id="package-participant", type="text", value="L000", placeholder="Participant ID", style={"width": "100%"}),
            dcc.Input(id="package-reviewer", type="text", placeholder="Reviewer initials", style={"width": "100%", "marginTop": "6px"}),
            dcc.Dropdown(id="processing-section-context", options=[{"label": value, "value": value} for value in PROCESSING_CONTEXTS], value="or1", clearable=False, style={"marginTop": "6px"}),
            dcc.Checklist(id="replace-existing-context", options=[{"label": "I confirm: replace existing context", "value": "replace"}], value=[], style={"marginTop": "6px", "fontSize": "11px"}),
            html.Button("Confirm, z-score, add to patient package", id="confirm-package-interval", n_clicks=0, style={"marginTop": "6px", "fontWeight": "bold"}),
            html.Div(id="package-status", children="", style={"fontSize": "11px", "marginTop": "6px"}),
        ]),
    ], style={"display": "grid", "gridTemplateColumns": "repeat(4, minmax(0, 1fr))", "alignItems": "start", "gap": "10px", "marginTop": "10px"}),
    dash_table.DataTable(id="manual-annotation-table", columns=[], data=[], style_table={"display": "none"}),
    dash_table.DataTable(id="processing-section-table", columns=[], data=[], style_table={"display": "none"}),
    html.Button(id="add-processing-section", n_clicks=0, style={"display": "none"}),
    html.Button(id="undo-processing-section", n_clicks=0, style={"display": "none"}),
    html.Div(id="processing-section-status", style={"display": "none"}),
], style={"fontFamily": "Arial, sans-serif", "padding": "10px", "maxWidth": "100vw", "boxSizing": "border-box"})

def _package_focused_interval(participant, reviewer, context, allow_replace=False, robust_z_threshold=3.0):
    """Write the focused interval, or the full recording when none is focused."""
    global STEP4_SYNCED_COUNT
    participant = str(participant or "").strip().upper()
    reviewer = str(reviewer or "").strip()
    if not __import__("re").fullmatch(r"L[0-9]{3,}", participant):
        raise ValueError("Participant must be deidentified, for example L0123.")
    if not __import__("re").fullmatch(r"[A-Za-z0-9_-]{1,20}", reviewer):
        raise ValueError("Reviewer ID must be 1–20 letters, numbers, _ or -.")
    for annotation in STEP4_MANUAL_ANNOTATIONS[STEP4_SYNCED_COUNT:]:
        channels = [] if "ALL" in annotation["channels"] else list(annotation["channels"])
        for raw_key in ("master", "filtered", "display"):
            review[raw_key].annotations.append(onset=[float(annotation["time_sec"])], duration=[float(annotation["duration_sec"])], description=[str(annotation["label"])], ch_names=[channels])
    STEP4_SYNCED_COUNT = len(STEP4_MANUAL_ANNOTATIONS)

    filtered = review["filtered"].copy().load_data()
    master = review["master"]
    sampling_rate = float(filtered.info["sfreq"])
    if STEP4_FOCUSED_INTERVAL:
        start_sec = float(STEP4_FOCUSED_INTERVAL["start_sec"])
        end_sec = float(STEP4_FOCUSED_INTERVAL["end_sec"])
    else:
        start_sec = 0.0
        end_sec = float(filtered.n_times / sampling_rate)
    start_sample = max(0, int(round(start_sec * sampling_rate)))
    end_sample = min(filtered.n_times, int(round(end_sec * sampling_rate)))
    data = filtered.get_data(start=start_sample, stop=end_sample).astype(float)
    if data.shape[1] == 0:
        raise ValueError("Selected interval has no samples.")
    mean, std, fit_channels, fit_points = recording_wide_global_scale(robust_z_threshold)
    if not np.isfinite(std) or std == 0:
        raise ValueError("Recording-wide global standard deviation is invalid.")
    zdata = ((data - mean) / std).astype(np.float32)

    export_folder = Path(globals().get("EXPORT_FOLDER", Path.cwd() / "reviewed_eeg"))
    export_folder.mkdir(parents=True, exist_ok=True)
    package = export_folder / f"{participant}_reviewed_eeg.h5"
    text_type = h5py.string_dtype(encoding="utf-8")
    with h5py.File(package, "a") as h5:
        h5.attrs["format_name"] = "Anesthesia EEG participant package"
        h5.attrs["participant_id"] = participant
        recordings = h5.require_group("recordings")
        if context in recordings:
            if not allow_replace:
                raise ValueError(f"{context} is already in this patient package. Tick replacement confirmation to replace it.")
            del recordings[context]
        recording = recordings.create_group(context)
        recording.attrs["source_start_sec"] = start_sec
        recording.attrs["source_end_sec"] = end_sec
        recording.attrs["reviewer"] = reviewer
        recording.attrs["packaged_utc"] = datetime.now(timezone.utc).isoformat()
        eeg_group = recording.create_group("eeg")
        eeg_group.create_dataset("data_average_referenced_global_zscore", data=zdata, compression="gzip", compression_opts=4, shuffle=True, chunks=(1, max(1, min(zdata.shape[1], int(sampling_rate * 10)))))
        eeg_group.attrs["sampling_frequency_hz"] = sampling_rate
        eeg_group.attrs["reference"] = "common average across all EEG channels"
        eeg_group.attrs["normalization"] = "recording-wide global z-score fit on global good channels and non-BAD one-second points"
        eeg_group.attrs["global_mean_volts"] = mean
        eeg_group.attrs["global_std_volts"] = std
        eeg_group.attrs["recording_wide_robust_z_fit_threshold"] = float(robust_z_threshold)
        eeg_group.attrs["recording_wide_global_zscore_fit_channels"] = fit_channels
        eeg_group.attrs["recording_wide_global_zscore_fit_points"] = fit_points
        eeg_group.create_dataset("channel_labels", data=np.asarray(master.ch_names, dtype=object), dtype=text_type)
        eeg_group.create_dataset("globally_bad_channels", data=np.asarray(master.info["bads"], dtype=object), dtype=text_type)
        annotations = recording.create_group("annotations")
        rows=[]
        for onset, duration, label, channels in zip(master.annotations.onset, master.annotations.duration, master.annotations.description, master.annotations.ch_names):
            onset, duration = float(onset), float(duration); end = onset + duration
            if (duration == 0 and not start_sec <= onset < end_sec) or (duration > 0 and (end <= start_sec or onset >= end_sec)):
                continue
            clipped = max(onset, start_sec)
            rows.append((clipped-start_sec, 0.0 if duration == 0 else min(end, end_sec)-clipped, str(label), ",".join(channels) if channels else "all"))
        annotations.create_dataset("onset_sec", data=np.asarray([r[0] for r in rows], dtype=float))
        annotations.create_dataset("duration_sec", data=np.asarray([r[1] for r in rows], dtype=float))
        annotations.create_dataset("label", data=np.asarray([r[2] for r in rows], dtype=object), dtype=text_type)
        annotations.create_dataset("channels", data=np.asarray([r[3] for r in rows], dtype=object), dtype=text_type)
        h5.attrs["n_recordings"] = len(recordings)
    return package, False


@step4_app.callback(
    Output("eeg-graph", "figure", allow_duplicate=True),
    Output("zscore-preview-status", "children"),
    Input("preview-zscore", "n_clicks"),
    Input("show-average-reference", "n_clicks"),
    State("high-variance-threshold", "value"),
    prevent_initial_call=True,
)
def preview_focused_zscore(preview_clicks, physical_clicks, robust_z_threshold):
    global data_to_plot, DISPLAY_AMPLITUDE_LIMIT_UV, CHANNEL_SPACING, offsets, STEP4_DISPLAY_MODE
    if ctx.triggered_id == "show-average-reference":
        data_to_plot = PHYSICAL_DATA_TO_PLOT.copy()
        DISPLAY_AMPLITUDE_LIMIT_UV = float(np.nanpercentile(np.abs(data_to_plot), 98.0)) or 1.0
        CHANNEL_SPACING = 2.5 * DISPLAY_AMPLITUDE_LIMIT_UV
        offsets = np.arange(number_channels)[::-1] * CHANNEL_SPACING
        STEP4_DISPLAY_MODE = "average-referenced µV"
        return build_interactive_eeg_figure(), "Showing average-referenced signal."
    filtered = review["filtered"].copy().load_data()
    # Whole-recording preview intentionally uses global bad channels only.
    # Focused-interval bad-channel annotations apply when that interval is saved.
    if not [name for name in filtered.ch_names if name not in review["master"].info["bads"]]:
        return no_update, "No global good channels remain for the z-score preview."
    mean, std, good_count, fit_sample_count = recording_wide_global_scale(robust_z_threshold or 3.0)
    if not np.isfinite(std) or std == 0:
        return no_update, "Cannot create z-score preview: invalid standard deviation."
    all_zscore = (filtered.get_data().astype(float) - mean) / std
    data_to_plot = all_zscore[:, display_samples].astype(np.float32, copy=False)
    # Fixed z-score scale makes changes in the good-channel fit visible;
    # do not auto-rescale each preview to its own percentile.
    DISPLAY_AMPLITUDE_LIMIT_UV = 5.0
    data_to_plot = np.clip(data_to_plot, -DISPLAY_AMPLITUDE_LIMIT_UV, DISPLAY_AMPLITUDE_LIMIT_UV)
    CHANNEL_SPACING = 2.5 * DISPLAY_AMPLITUDE_LIMIT_UV
    offsets = np.arange(number_channels)[::-1] * CHANNEL_SPACING
    STEP4_DISPLAY_MODE = "global z-score"
    return build_interactive_eeg_figure(), f"Previewing recording-wide global z-score from {good_count} good channels and {fit_sample_count} one-second fit points (threshold ±{float(robust_z_threshold or 3.0):g}). Confirm to package this focused interval."


@step4_app.callback(
    Output("eeg-graph", "figure", allow_duplicate=True),
    Output("action-status", "children", allow_duplicate=True),
    Input("toggle-full-rate-view", "n_clicks"),
    State("eeg-graph", "relayoutData"),
    prevent_initial_call=True,
)
def toggle_full_rate_view(clicks, relayout_data):
    global STEP4_HIGH_RES_ACTIVE, STEP4_VIEW_START_MINUTE, STEP4_VIEW_END_MINUTE
    if STEP4_HIGH_RES_ACTIVE:
        STEP4_HIGH_RES_ACTIVE = False
        STEP4_VIEW_START_MINUTE = None
        STEP4_VIEW_END_MINUTE = None
        set_adaptive_display_window()
        return build_interactive_eeg_figure(), "Returned to the fast 20,000-point overview."
    if STEP4_DISPLAY_MODE == "global z-score":
        return no_update, "Switch to average-reference view before requesting 250 Hz detail."
    if not isinstance(relayout_data, dict):
        return no_update, "Zoom to a short interval first, then switch to 250 Hz view."
    try:
        left = float(relayout_data.get("xaxis.range[0]"))
        right = float(relayout_data.get("xaxis.range[1]"))
    except (TypeError, ValueError):
        return no_update, "Zoom to a short interval first, then switch to 250 Hz view."
    seconds = (right - left) * 60.0
    if seconds <= 0 or seconds > ADAPTIVE_FULL_RES_MAX_SECONDS:
        return no_update, f"Choose a range no longer than {ADAPTIVE_FULL_RES_MAX_SECONDS:.0f} seconds for 250 Hz review."
    STEP4_VIEW_START_MINUTE, STEP4_VIEW_END_MINUTE = left, right
    STEP4_HIGH_RES_ACTIVE = True
    set_adaptive_display_window(left, right)
    return build_interactive_eeg_figure(), f"Showing {seconds:.1f} sec at 250 Hz. Click again to return to overview."


@step4_app.callback(
    Output("replace-annotation-status", "children"),
    Output("replace-annotation-index", "options"),
    Output("eeg-graph", "figure", allow_duplicate=True),
    Input("replace-annotation", "n_clicks"),
    Input("delete-annotation", "n_clicks"),
    State("replace-annotation-index", "value"),
    State("replace-annotation-label", "value"),
    prevent_initial_call=True,
)
def replace_or_delete_annotation(replace_clicks, delete_clicks, selected_index, standardized_label):
    global eeg, eeg_display
    if selected_index is None:
        return "Choose an existing annotation first.", no_update, no_update
    index = int(selected_index)
    raw_master = review["master"]
    if not 0 <= index < len(raw_master.annotations):
        return "That annotation is no longer available; refresh Step 4.", no_update, no_update
    old_label = str(raw_master.annotations.description[index])
    deleting = ctx.triggered_id == "delete-annotation"
    for raw_key in ("master", "filtered", "display"):
        raw = review[raw_key]
        onsets = list(raw.annotations.onset)
        durations = list(raw.annotations.duration)
        descriptions = list(raw.annotations.description)
        channel_sets = list(raw.annotations.ch_names)
        if deleting:
            for values in (onsets, durations, descriptions, channel_sets):
                values.pop(index)
        else:
            descriptions[index] = str(standardized_label)
        raw.set_annotations(mne.Annotations(
            onset=onsets, duration=durations, description=descriptions,
            ch_names=channel_sets, orig_time=raw.annotations.orig_time,
        ))
    eeg = sync_legacy_eeg_dictionary(review, eeg)
    eeg_display = make_plotly_display_eeg(review, eeg)
    options = [
        {"label": f"{float(onset):.3f} sec — {description}", "value": str(item_index)}
        for item_index, (onset, description) in enumerate(zip(review["master"].annotations.onset, review["master"].annotations.description))
    ]
    status = f"Deleted {old_label}." if deleting else f"Replaced {old_label} with {standardized_label}."
    return status, options, build_interactive_eeg_figure()


@step4_app.callback(
    Output("batch-delete-status", "children"),
    Output("replace-annotation-index", "options", allow_duplicate=True),
    Output("eeg-graph", "figure", allow_duplicate=True),
    Output("batch-delete-selection", "options"),
    Output("batch-delete-selection", "value"),
    Input("batch-delete-find", "n_clicks"), Input("batch-delete-confirm", "n_clicks"),
    State("batch-delete-search", "value"), State("batch-delete-ok", "value"), State("batch-delete-selection", "value"),
    prevent_initial_call=True,
)
def find_or_delete_annotation_batch(find_clicks, delete_clicks, search_text, confirmed, selected_indices):
    global eeg, eeg_display
    labels = [str(value) for value in review["master"].annotations.description]
    query = str(search_text or "").strip().casefold()
    if query:
        matches = [index for index, value in enumerate(labels) if query in value.casefold()]
        description = f"Found {len(matches)} labels containing '{search_text}'."
    else:
        counts = {}
        for value in labels:
            key = value.casefold()
            counts[key] = counts.get(key, 0) + 1
        matches = [index for index, value in enumerate(labels) if counts[value.casefold()] > 1]
        description = f"Suggested {len(matches)} annotations with duplicate labels."
    options = [{"label": f"{index}: {float(review['master'].annotations.onset[index]):.3f} sec — {labels[index]}", "value": str(index)} for index in matches]
    if ctx.triggered_id == "batch-delete-find":
        return description + " Select the entries to delete, then tick confirmation.", no_update, no_update, options, [str(index) for index in matches]
    selected = {int(index) for index in (selected_indices or []) if str(index).isdigit()}
    selected &= set(matches)
    if not selected:
        return description + " Select at least one displayed entry before deleting.", no_update, no_update, options, no_update
    if "yes" not in (confirmed or []):
        return f"{len(selected)} entries selected. Tick confirmation before deleting.", no_update, no_update, options, no_update
    for raw_key in ("master", "filtered", "display"):
        raw = review[raw_key]
        keep = [index for index in range(len(raw.annotations)) if index not in selected]
        raw.set_annotations(mne.Annotations(onset=[raw.annotations.onset[index] for index in keep], duration=[raw.annotations.duration[index] for index in keep], description=[raw.annotations.description[index] for index in keep], ch_names=[raw.annotations.ch_names[index] for index in keep], orig_time=raw.annotations.orig_time))
    eeg = sync_legacy_eeg_dictionary(review, eeg); eeg_display = make_plotly_display_eeg(review, eeg)
    annotation_options = [{"label": f"{float(onset):.3f} sec — {description}", "value": str(index)} for index, (onset, description) in enumerate(zip(review["master"].annotations.onset, review["master"].annotations.description))]
    return f"Deleted {len(selected)} selected annotations.", annotation_options, build_interactive_eeg_figure(), [], []


@step4_app.callback(
    Output("bad-channel-status", "children"),
    Output("bad-channel-selection", "value"),
    Output("eeg-graph", "figure", allow_duplicate=True),
    Input("set-bad-channels", "n_clicks"),
    Input("clear-bad-channels", "n_clicks"),
    State("bad-channel-selection", "value"),
    State("bad-channel-scope", "value"),
    prevent_initial_call=True,
)
def set_globally_bad_channels(set_clicks, clear_clicks, selected, scope):
    bads = [] if ctx.triggered_id == "clear-bad-channels" else list(selected or [])
    if scope == "focused" and ctx.triggered_id == "set-bad-channels":
        if not STEP4_FOCUSED_INTERVAL:
            return "Focus an interval before marking interval-specific bad channels.", no_update, no_update
        annotation = {"time_sec": round(STEP4_FOCUSED_INTERVAL["start_sec"], 3), "duration_sec": round(STEP4_FOCUSED_INTERVAL["end_sec"] - STEP4_FOCUSED_INTERVAL["start_sec"], 3), "label": "BAD_channel", "channels": bads}
        STEP4_MANUAL_ANNOTATIONS.append(annotation)
        return "Marked selected channels BAD_channel in the focused interval.", bads, build_interactive_eeg_figure()
    for raw_key in ("master", "filtered", "display"):
        review[raw_key].info["bads"] = list(bads)
    if ctx.triggered_id == "clear-bad-channels":
        return "Global bad-channel designation cleared; normal display restored.", [], build_interactive_eeg_figure()
    return "Globally bad channels: " + (", ".join(bads) if bads else "none"), bads, build_interactive_eeg_figure()


# Rename labels in every raw object so the Step 5 file uses the new names.
@step4_app.callback(
    Output("eeg-graph", "figure", allow_duplicate=True),
    Output("affected-channels", "options"),
    Output("channel-rename-source", "options"),
    Output("channel-rename-status", "children"),
    Input("apply-channel-rename", "n_clicks"),
    State("channel-rename-source", "value"),
    State("channel-rename-target", "value"),
    prevent_initial_call=True,
)
def rename_channel_for_recording(clicks, old_name, new_name):
    global channel_labels, channel_options
    if not old_name or not new_name or old_name not in channel_labels:
        return no_update, no_update, no_update, "Choose a current channel and a replacement."
    if old_name == new_name:
        return no_update, no_update, no_update, "That channel already has this name."
    if new_name in channel_labels:
        return no_update, no_update, no_update, f"{new_name} is already assigned."
    for raw_key in ("master", "filtered", "display"):
        review[raw_key].rename_channels({old_name: new_name})
    channel_labels[channel_labels.index(old_name)] = new_name
    for annotation in STEP4_MANUAL_ANNOTATIONS:
        annotation["channels"] = [new_name if value == old_name else value for value in annotation["channels"]]
    channel_options = [{"label": "ALL channels", "value": "ALL"}] + [{"label": value, "value": value} for value in channel_labels]
    source_options = [{"label": value, "value": value} for value in channel_labels]
    return build_interactive_eeg_figure(), channel_options, source_options, f"Renamed {old_name} to {new_name} for the whole recording."




def detect_high_variance_candidates(threshold, density):
    """Detect 1-sec blocks by either dense high-amplitude points or high variance."""
    raw = review["filtered"].copy().load_data()
    values = raw.get_data().astype(float)
    amplitude = np.nanpercentile(np.abs(values), 75.0, axis=0)
    median = float(np.nanmedian(amplitude)); mad = float(np.nanmedian(np.abs(amplitude - median)))
    robust_sd = 1.4826 * mad
    point_flagged = amplitude > median + float(threshold) * robust_sd if robust_sd > 0 else np.zeros(amplitude.size, dtype=bool)
    sfreq = float(raw.info["sfreq"]); block = max(1, int(round(sfreq)))
    block_variance=[]
    for left in range(0, values.shape[1], block):
        right=min(values.shape[1],left+block)
        block_variance.append(float(np.nanmedian(np.nanstd(values[:,left:right],axis=1))))
    block_variance=np.asarray(block_variance)
    variance_median=float(np.nanmedian(block_variance)); variance_mad=float(np.nanmedian(np.abs(block_variance-variance_median)))
    variance_threshold=variance_median + float(threshold)*1.4826*variance_mad
    selected=np.zeros(point_flagged.size,dtype=bool); amplitude_blocks=0; variance_blocks=0
    for block_index,left in enumerate(range(0,point_flagged.size,block)):
        right=min(point_flagged.size,left+block)
        dense=point_flagged[left:right].mean() >= float(density)
        variable=block_variance[block_index] > variance_threshold if variance_mad > 0 else False
        if dense or variable:
            selected[left:right]=True
            amplitude_blocks += int(dense); variance_blocks += int(variable)
    globals()["STEP4_HIGH_VARIANCE_DIAGNOSTICS"]={"amplitude_blocks":amplitude_blocks,"variance_blocks":variance_blocks,"max_amplitude_robust_z":float((np.nanmax(amplitude)-median)/robust_sd) if robust_sd>0 else float("nan"),"max_variance_robust_z":float((np.nanmax(block_variance)-variance_median)/(1.4826*variance_mad)) if variance_mad>0 else float("nan")}
    runs=[]; run_start=None
    for index,value in enumerate(np.r_[selected,False]):
        if value and run_start is None: run_start=index
        if not value and run_start is not None:
            runs.append({"time_sec":round(run_start/sfreq,3),"duration_sec":round((index-run_start)/sfreq,3),"label":"BAD_high_variance","channels":["ALL"]}); run_start=None
    return runs


def recording_wide_global_scale(robust_z_threshold=3.0):
    """Fast recording-wide scale fit on one representative sample per second.

    The saved focused interval is still z-scored sample-by-sample.  Only the
    fitting set is downsampled, which keeps long recordings responsive.
    """
    threshold = float(robust_z_threshold or 3.0)
    if not np.isfinite(threshold) or threshold <= 0:
        raise ValueError("Robust z-score threshold must be a positive number.")
    sample_step = max(1, int(round(sampling_rate)))
    fit_indices = np.arange(0, number_samples, sample_step, dtype=int)
    values = data_filtered[:, fit_indices].astype(float, copy=False)
    good = [index for index, name in enumerate(channel_labels) if name not in review["master"].info["bads"]]
    if not good:
        raise ValueError("No global good channels remain for recording-wide normalization.")

    fit_times_sec = fit_indices / float(sampling_rate)
    keep = np.ones(fit_indices.size, dtype=bool)
    for onset, duration, label in zip(review["master"].annotations.onset, review["master"].annotations.duration, review["master"].annotations.description):
        if str(label).startswith("BAD_") and str(label) != "BAD_channel":
            keep &= ~((fit_times_sec >= float(onset)) & (fit_times_sec < float(onset) + float(duration)))
    for annotation in STEP4_MANUAL_ANNOTATIONS:
        if str(annotation["label"]).startswith("BAD_") and str(annotation["label"]) != "BAD_channel":
            keep &= ~((fit_times_sec >= float(annotation["time_sec"])) & (fit_times_sec < float(annotation["time_sec"]) + float(annotation["duration_sec"])))

    preliminary = values[good][:, keep] if keep.any() else values[good]
    robust_center = float(np.nanmedian(preliminary))
    robust_sd = 1.4826 * float(np.nanmedian(np.abs(preliminary - robust_center)))
    automatic_rejected = np.zeros(keep.size, dtype=bool)
    if np.isfinite(robust_sd) and robust_sd > 0:
        point_score = np.nanpercentile(np.abs((values[good] - robust_center) / robust_sd), 75.0, axis=0)
        automatic_rejected = point_score > threshold
        keep &= ~automatic_rejected
    if not keep.any():
        raise ValueError("No downsampled good points remain after robust-z exclusion.")
    fit = values[good][:, keep]
    globals()["STEP4_RECORDING_WIDE_THREE_Z_REJECTED"] = int(automatic_rejected.sum())
    globals()["STEP4_RECORDING_WIDE_THREE_Z_FIT_SAMPLES"] = int(keep.sum())
    globals()["STEP4_RECORDING_WIDE_SCALE_SAMPLE_STEP_SEC"] = float(sample_step / sampling_rate)
    return float(np.nanmean(fit)), float(np.nanstd(fit)), len(good), int(keep.sum())


@step4_app.callback(
    Output("eeg-graph", "figure", allow_duplicate=True),
    Output("package-status", "children"),
    Output("replace-existing-context", "value"),
    Input("confirm-package-interval", "n_clicks"),
    State("package-participant", "value"),
    State("package-reviewer", "value"),
    State("processing-section-context", "value"),
    State("replace-existing-context", "value"),
    State("high-variance-threshold", "value"),
    prevent_initial_call=True,
)
def confirm_and_package_interval(clicks, participant, reviewer, context, replace_existing, robust_z_threshold):
    global STEP4_VIEW_START_MINUTE, STEP4_VIEW_END_MINUTE, STEP4_FOCUSED_INTERVAL
    global data_to_plot, DISPLAY_AMPLITUDE_LIMIT_UV, CHANNEL_SPACING, offsets, STEP4_DISPLAY_MODE
    try:
        output, used_all_samples_for_scale = _package_focused_interval(participant, reviewer, str(context), allow_replace="replace" in (replace_existing or []), robust_z_threshold=robust_z_threshold or 3.0)
    except Exception as error:
        return no_update, f"Not packaged: {error}", no_update
    STEP4_VIEW_START_MINUTE = None
    STEP4_VIEW_END_MINUTE = None
    STEP4_FOCUSED_INTERVAL = None
    data_to_plot = PHYSICAL_DATA_TO_PLOT.copy()
    DISPLAY_AMPLITUDE_LIMIT_UV = float(np.nanpercentile(np.abs(data_to_plot), 98.0)) or 1.0
    CHANNEL_SPACING = 2.5 * DISPLAY_AMPLITUDE_LIMIT_UV
    offsets = np.arange(number_channels)[::-1] * CHANNEL_SPACING
    STEP4_DISPLAY_MODE = "average-referenced µV"
    STEP4_PROCESSING_SEGMENTS[:] = []
    scale_note = " All time samples were used for scaling because the interval is entirely marked BAD; globally and interval-specific bad channels were still excluded." if used_all_samples_for_scale else ""
    replaced_note = " Replaced the previous context." if "replace" in (replace_existing or []) else ""
    return build_interactive_eeg_figure(), f"Added {context} to {output.name}.{replaced_note} Returned to full recording; select the next interval.{scale_note}", []


# ---------------------------------------------------------
# Capture interval or point selections.
# ---------------------------------------------------------

@step4_app.callback(
    Output(
        "start-minutes",
        "value",
    ),
    Output(
        "end-minutes",
        "value",
    ),
    Output(
        "selection-status",
        "children",
    ),
    Input(
        "eeg-graph",
        "selectedData",
    ),
    Input(
        "eeg-graph",
        "clickData",
    ),
    Input(
        "annotation-kind",
        "value",
    ),
    prevent_initial_call=True,
)
def capture_plot_selection(
    selected_data,
    click_data,
    annotation_kind,
):

    if annotation_kind == "interval":

        if not selected_data:

            return (
                no_update,
                no_update,
                (
                    "Drag a box over the "
                    "desired interval."
                ),
            )

        # Plotly normally provides the exact
        # box-selection x-axis range.
        selected_range = selected_data.get(
            "range",
            {},
        ).get(
            "x",
        )

        if selected_range:

            start_minute = float(
                min(selected_range)
            )

            end_minute = float(
                max(selected_range)
            )

        else:

            points = selected_data.get(
                "points",
                [],
            )

            if not points:

                return (
                    no_update,
                    no_update,
                    "No EEG points selected.",
                )

            selected_times = [
                float(point["x"])
                for point in points
            ]

            start_minute = min(
                selected_times
            )

            end_minute = max(
                selected_times
            )

        start_minute = round(
            start_minute,
            4,
        )

        end_minute = round(
            end_minute,
            4,
        )

        duration_minute = round(
            end_minute - start_minute,
            4,
        )

        return (
            start_minute,
            end_minute,
            (
                f"Selected interval: "
                f"{start_minute:.4f}–"
                f"{end_minute:.4f} minutes "
                f"(duration "
                f"{duration_minute:.4f} min)"
            ),
        )

    # Point-event mode.
    if not click_data:

        return (
            no_update,
            no_update,
            "Click the desired event time.",
        )

    clicked_minute = float(
        click_data["points"][0]["x"]
    )

    onset_minute = round(
        clicked_minute,
        4,
    )

    return (
        onset_minute,
        onset_minute,
        (
            f"Selected point: "
            f"{onset_minute:.4f} minutes"
        ),
    )


# ---------------------------------------------------------
# Add and undo annotations.
# ---------------------------------------------------------

@step4_app.callback(
    Output(
        "eeg-graph",
        "figure",
    ),
    Output(
        "manual-annotation-table",
        "data",
    ),
    Output(
        "action-status",
        "children",
    ),
    Input(
        "add-annotation",
        "n_clicks",
    ),
    Input(
        "undo-annotation",
        "n_clicks",
    ),
    Input("decrease-display-gain", "n_clicks"),
    Input("increase-display-gain", "n_clicks"),
    Input("reset-display-gain", "n_clicks"),
    Input("focus-selected-interval", "n_clicks"),
    Input("show-full-recording", "n_clicks"),
    Input("clear-focus-selection", "n_clicks"),
    State(
        "annotation-kind",
        "value",
    ),
    State(
        "standard-label",
        "value",
    ),
    State(
        "affected-channels",
        "value",
    ),
    State(
        "start-minutes",
        "value",
    ),
    State(
        "end-minutes",
        "value",
    ),
    prevent_initial_call=True,
)
def modify_manual_annotations(
    add_clicks,
    undo_clicks,
    decrease_gain_clicks,
    increase_gain_clicks,
    reset_gain_clicks,
    focus_clicks,
    show_full_clicks,
    clear_focus_clicks,
    annotation_kind,
    standard_label,
    affected_channels,
    start_minute,
    end_minute,
):

    global STEP4_DISPLAY_GAIN, STEP4_VIEW_START_MINUTE, STEP4_VIEW_END_MINUTE, STEP4_FOCUSED_INTERVAL, STEP4_DISPLAY_MODE, STEP4_HIGH_RES_ACTIVE
    triggered = ctx.triggered_id

    if triggered == "focus-selected-interval":
        if start_minute is None or end_minute is None or float(end_minute) <= float(start_minute):
            return no_update, no_update, "Select a non-zero interval before focusing it."
        STEP4_VIEW_START_MINUTE = float(start_minute)
        STEP4_VIEW_END_MINUTE = float(end_minute)
        STEP4_FOCUSED_INTERVAL = {
            "start_sec": STEP4_VIEW_START_MINUTE * 60.0,
            "end_sec": STEP4_VIEW_END_MINUTE * 60.0,
        }
        set_adaptive_display_window(STEP4_VIEW_START_MINUTE, STEP4_VIEW_END_MINUTE)
        return build_interactive_eeg_figure(), no_update, (
            f"Focused view: {STEP4_VIEW_START_MINUTE:.4f}–{STEP4_VIEW_END_MINUTE:.4f} minutes."
        )
    if triggered == "show-full-recording":
        STEP4_VIEW_START_MINUTE = None
        STEP4_VIEW_END_MINUTE = None
        set_adaptive_display_window()
        return build_interactive_eeg_figure(), no_update, "Showing the full recording overview."

    if triggered == "clear-focus-selection":
        STEP4_FOCUSED_INTERVAL = None
        return no_update, no_update, "Focused interval cleared. Packaging will use the whole recording unless you focus a new interval."

    if triggered == "decrease-display-gain":
        STEP4_DISPLAY_GAIN = max(0.01, STEP4_DISPLAY_GAIN / 2.0)
        return build_interactive_eeg_figure(), no_update, f"Display gain: {STEP4_DISPLAY_GAIN:.2f}×"
    if triggered == "increase-display-gain":
        STEP4_DISPLAY_GAIN = STEP4_DISPLAY_GAIN * 2.0
        return build_interactive_eeg_figure(), no_update, f"Display gain: {STEP4_DISPLAY_GAIN:.2f}×"
    if triggered == "reset-display-gain":
        STEP4_DISPLAY_GAIN = 1.0
        STEP4_HIGH_RES_ACTIVE = False
        STEP4_VIEW_START_MINUTE = None
        STEP4_VIEW_END_MINUTE = None
        STEP4_DISPLAY_MODE = "average-referenced µV"
        set_adaptive_display_window()
        return build_interactive_eeg_figure(), no_update, "Display reset: full average-referenced recording at 1.00× gain."

    if triggered == "undo-annotation":

        if STEP4_MANUAL_ANNOTATIONS:

            removed = (
                STEP4_MANUAL_ANNOTATIONS.pop()
            )

            status = (
                "Removed last annotation: "
                f"{removed['label']}"
            )

        else:

            status = (
                "There are no new "
                "annotations to undo."
            )

    elif triggered == "add-annotation":

        if start_minute is None:

            return (
                no_update,
                no_update,
                "Select a time first.",
            )

        if not affected_channels:

            return (
                no_update,
                no_update,
                "Select ALL or affected channels.",
            )

        start_minute = float(
            start_minute
        )

        if annotation_kind == "interval":

            if end_minute is None:

                return (
                    no_update,
                    no_update,
                    "Select an interval end time.",
                )

            end_minute = float(
                end_minute
            )

            if end_minute <= start_minute:

                return (
                    no_update,
                    no_update,
                    (
                        "Interval end must be "
                        "after interval start."
                    ),
                )

            duration_minute = (
                end_minute - start_minute
            )

        else:

            duration_minute = 0.0

        # MNE and the saved HDF5 use seconds internally.
        # Only the Step 4 entry boxes use minutes.
        start_sec = start_minute * 60.0
        duration_sec = duration_minute * 60.0

        if "ALL" in affected_channels:

            saved_channels = ["ALL"]

        else:

            saved_channels = list(
                affected_channels
            )

        annotation = {
            "time_sec": round(
                start_sec,
                3,
            ),
            "duration_sec": round(
                duration_sec,
                3,
            ),
            "label": str(
                standard_label
            ),
            "channels": saved_channels,
        }

        STEP4_MANUAL_ANNOTATIONS.append(
            annotation
        )

        status = (
            f"Added {standard_label} "
            f"at {start_minute:.4f} minutes."
        )

    else:

        return (
            no_update,
            no_update,
            no_update,
        )

    table_rows = []

    for annotation in (
        STEP4_MANUAL_ANNOTATIONS
    ):

        table_rows.append(
            {
                "time_sec": (
                    annotation["time_sec"]
                ),
                "duration_sec": (
                    annotation[
                        "duration_sec"
                    ]
                ),
                "label": annotation["label"],
                "channels_text": ",".join(
                    annotation["channels"]
                ),
            }
        )

    return (
        build_interactive_eeg_figure(),
        table_rows,
        status,
    )


# ---------------------------------------------------------
# Add and remove processing sections selected in the graph.
# ---------------------------------------------------------

@step4_app.callback(
    Output("processing-section-table", "data"),
    Output("processing-section-status", "children"),
    Input("add-processing-section", "n_clicks"),
    Input("undo-processing-section", "n_clicks"),
    State("processing-section-context", "value"),
    State("start-minutes", "value"),
    State("end-minutes", "value"),
    prevent_initial_call=True,
)
def modify_processing_sections(add_clicks, undo_clicks, context, start_minute, end_minute):
    triggered = ctx.triggered_id
    if triggered == "undo-processing-section":
        if STEP4_PROCESSING_SEGMENTS:
            removed = STEP4_PROCESSING_SEGMENTS.pop()
            status = f"Cleared {removed['context']} selected interval."
        else:
            status = "There are no processing sections to undo."
    else:
        if start_minute is None or end_minute is None:
            return no_update, "Select an interval in the graph before adding a processing section."
        start_sec = round(float(start_minute) * 60.0, 3)
        end_sec = round(float(end_minute) * 60.0, 3)
        if end_sec <= start_sec:
            return no_update, "A processing section needs a non-zero interval."
        # One Step 5 export is one reviewed interval. Replacing the prior
        # selection lets the reviewer correct the choice without reloading Step 4.
        STEP4_PROCESSING_SEGMENTS[:] = []
        STEP4_PROCESSING_SEGMENTS.append({
            "context": str(context),
            "start_sec": start_sec,
            "end_sec": end_sec,
            "duration_sec": round(end_sec - start_sec, 3),
        })
        status = f"Selected {context}: {start_sec:.3f}–{end_sec:.3f} sec for Step 5."
    return list(STEP4_PROCESSING_SEGMENTS), status


# ---------------------------------------------------------
# Find an available local port and launch the browser app without blocking the notebook.
# ---------------------------------------------------------

def stop_step4_dashboard():
    """Stop the local Step 4 server when the review moves to Step 5."""
    server = globals().get("STEP4_DASH_SERVER")
    if server is not None:
        server.shutdown()
        server.server_close()
        globals()["STEP4_DASH_SERVER"] = None


stop_step4_dashboard()

with socket.socket() as port_socket:

    port_socket.bind(
        ("127.0.0.1", 0)
    )

    STEP4_DASH_PORT = int(
        port_socket.getsockname()[1]
    )

STEP4_DASH_URL = (
    f"http://127.0.0.1:"
    f"{STEP4_DASH_PORT}"
)

print(
    "Starting interactive EEG annotation interface:"
)

print(
    STEP4_DASH_URL
)

# Open the browser shortly after the server begins.
threading.Timer(
    1.2,
    lambda: webbrowser.open(
        STEP4_DASH_URL
    ),
).start()

STEP4_DASH_SERVER = make_server(
    "127.0.0.1",
    STEP4_DASH_PORT,
    step4_app.server,
    threaded=True,
)
STEP4_DASH_THREAD = threading.Thread(
    target=STEP4_DASH_SERVER.serve_forever,
    daemon=True,
)
STEP4_DASH_THREAD.start()

print(
    "Step 4 is running in the browser and this cell is complete. "
    "Keep the notebook kernel running; Step 5 will stop the local server."
)

Display filtering: 0.5 to 50.0 Hz
Starting interactive EEG annotation interface:
http://127.0.0.1:59264
Step 4 is running in the browser and this cell is complete. Keep the notebook kernel running; Step 5 will stop the local server.


INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 10:26:05] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 10:26:05] "GET /_dash-dependencies HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 10:26:05] "GET /_dash-layout HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 10:26:06] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 10:26:06] "GET /_dash-component-suites/dash/dcc/async-dropdown.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 10:26:06] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 10:26:06] "GET /_dash-component-suites/plotly/package_data/plotly.min.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 10:26:06] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 10:26:15] "POST /_dash-update-component?endId=2muUPZXDVwQsrYeFfJ8v4R5Z~992f4